# Question 2: Chunking Strategy Showdown
Chunking is one of the most underappreciated decisions in RAG. In this question
you will empirically verify that claim by comparing three chunking strategies and
measuring their effect on retrieval quality.
Input: Use the same document from Q1, or replace it with a longer one (3,000+
words). You need enough content for the differences between strategies to show
up.


**Part A: Implement Three Chunkers**

Implement all three of the following. Each function must return a list of strings.

In [3]:
import re
import numpy as np

In [4]:


def fixed_chunk(text: str, size: int = 300, overlap: int = 50) -> list[str]:
    """Splits text into chunks of `size` words, with `overlap` words shared."""
    words = text.split()
    chunks = []
    step = size - overlap

    for i in range(0, len(words), step):
        chunk_words = words[i:i + size]
        chunks.append(" ".join(chunk_words))
        # Stop if we've reached the end of the text
        if i + size >= len(words):
            break

    return chunks

def sentence_chunk(text: str) -> list[str]:
    """Splits text into chunks of ~5 sentences with a 1-sentence overlap."""
    # Use regex positive lookbehind to split on punctuation followed by a space
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks = []

    group_size = 5
    overlap = 1
    step = group_size - overlap

    for i in range(0, len(sentences), step):
        chunk_sentences = sentences[i:i + group_size]
        chunks.append(" ".join(chunk_sentences))
        if i + group_size >= len(sentences):
            break

    return chunks

def sliding_window_chunk(text: str, window: int = 400, step: int = 100) -> list[str]:
    """Creates heavily overlapping chunks of `window` words, advancing by `step`."""
    words = text.split()
    chunks = []

    for i in range(0, len(words), step):
        chunk_words = words[i:i + window]
        chunks.append(" ".join(chunk_words))
        if i + window >= len(words):
            break

    return chunks

# --- Helper to print stats ---
def print_chunk_stats(strategy_name: str, chunks: list[str]):
    lengths = [len(c.split()) for c in chunks]
    print(f"{strategy_name: <15} | Chunks: {len(chunks): <3} | Mean Len: {np.mean(lengths):.0f} wds | Min: {np.min(lengths)}, Max: {np.max(lengths)}")

# --- Run the stats ---
# Assuming text_content is still loaded from your document in Q1
text_content = open("Document.txt", "r").read()

fixed_chunks_list = fixed_chunk(text_content)
sentence_chunks_list = sentence_chunk(text_content)
sliding_chunks_list = sliding_window_chunk(text_content)

print("-" * 65)
print_chunk_stats("Fixed-size", fixed_chunks_list)
print_chunk_stats("Sentence-based", sentence_chunks_list)
print_chunk_stats("Sliding window", sliding_chunks_list)
print("-" * 65)


-----------------------------------------------------------------
Fixed-size      | Chunks: 16  | Mean Len: 298 wds | Min: 276, Max: 300
Sentence-based  | Chunks: 48  | Mean Len: 104 wds | Min: 19, Max: 154
Sliding window  | Chunks: 38  | Mean Len: 398 wds | Min: 326, Max: 400
-----------------------------------------------------------------


**Part B: Retrieval Comparison**

To test this, we need to embed all three chunk lists separately, run our queries against them, and check if the exact string answer is found in the retrieved text.

Here is the evaluation block:

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# 1. Create the 5 Q&A pairs based on the text from Q1
qa_pairs = [
    {
        "query": "Who is the American psychologist that advances the importance of IQ?",
        "answer": "Daniel Goleman"
    },
    {
        "query": "What are the five elements of emotional intelligence?",
        "answer": "self-awareness, self-regulation, motivation, empathy, and social skills"
    },
    {
        "query": "Which pharmaceutical company in France studied emotional intelligence?",
        "answer": "Sanofi"
    },
    {
        "query": "What model can be integrated into all-inclusive management programmes?",
        "answer": "360 degrees feedback model"
    },
    {
        "query": "What type of leadership style issues orders without asking questions?",
        "answer": "autocratic"
    }
]

# 2. Reuse your cosine similarity function
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    norm_a, norm_b = np.linalg.norm(a), np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0: return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))

# 3. Define the evaluator function
def evaluate_strategy(chunks: list[str]) -> float:
    """Embeds the chunks, runs the 5 queries, and returns the hit rate."""
    chunk_embeddings = model.encode(chunks, convert_to_numpy=True)
    hits = 0

    for qa in qa_pairs:
        query_embedding = model.encode(qa["query"], convert_to_numpy=True)

        # Score all chunks
        scored_chunks = []
        for i, chunk_emb in enumerate(chunk_embeddings):
            score = cosine_similarity(query_embedding, chunk_emb)
            scored_chunks.append((chunks[i], score))

        # Sort and get top 3
        scored_chunks.sort(key=lambda x: x[1], reverse=True)
        top_3 = [chunk for chunk, score in scored_chunks[:3]]

        # Check for string match
        for context in top_3:
            # Case insensitive check
            if qa["answer"].lower() in context.lower():
                hits += 1
                break # Move to next query once a hit is found in top 3

    return hits / len(qa_pairs)

# 4. Run the evaluation and print the final formatted table
print(f"{'Strategy': <15} | {'Chunks': <6} | {'Mean Len': <8} | Hit Rate")
print("-" * 45)

strategies = [
    ("Fixed-size", fixed_chunks_list),
    ("Sentence-based", sentence_chunks_list),
    ("Sliding window", sliding_chunks_list)
]

for name, chunk_list in strategies:
    hit_rate = evaluate_strategy(chunk_list)
    mean_len = int(np.mean([len(c.split()) for c in chunk_list]))
    # Format the exact output requested
    print(f"{name: <15} | {len(chunk_list): <6} | {str(mean_len) + ' wds': <8} | {int(hit_rate*5)}/5")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Strategy        | Chunks | Mean Len | Hit Rate
---------------------------------------------
Fixed-size      | 16     | 298 wds  | 5/5
Sentence-based  | 48     | 104 wds  | 5/5
Sliding window  | 38     | 398 wds  | 5/5
